# EMA and MACD Technical Analysis (3-Year Period)

In this notebook, we analyze a selected stock (**TICKER**) using two widely used technical indicators:

- **Exponential Moving Averages (EMA)**: EMA20, EMA50, EMA200
- **MACD (Moving Average Convergence Divergence)**

These indicators help identify trends, momentum, and potential buy/sell signals.


In [ ]:
# --- Imports ---
from src.technical_setup import *

## Functions: EMA & MACD Calculations and Visualization

We define reusable functions to:
- Download price data
- Calculate EMA indicators
- Calculate MACD indicators
- Plot the results


In [ ]:
def download_price_data(ticker: str, period: str = "3y"):
    """
    Downloads historical price data from Yahoo Finance.

    Parameters
    ----------
    ticker : str
        Stock symbol (e.g., 'AAPL', 'MSFT').
    period : str, optional
        Time period to download (default is '3y').

    Returns
    -------
    pd.DataFrame
        DataFrame containing OHLCV price data.
    """
    df = yf.download(ticker, period=period)
    df.index = df.index.tz_localize(None)
    return df


In [ ]:
def calculate_ema(df, spans=(20, 50, 200)):
    """
    Calculates Exponential Moving Averages (EMA) for given spans.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing price data with a 'Close' column.
    spans : tuple, optional
        EMA spans to calculate (default is (20, 50, 200)).

    Returns
    -------
    pd.DataFrame
        DataFrame with EMA columns added.
    """
    for span in spans:
        df[f"EMA{span}"] = df["Close"].ewm(span=span, adjust=False).mean()
    return df


In [ ]:
def calculate_macd(df, fast=12, slow=26, signal=9):
    """
    Calculates MACD, Signal line, and Histogram.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing price data with a 'Close' column.
    fast : int, optional
        Fast EMA span (default is 12).
    slow : int, optional
        Slow EMA span (default is 26).
    signal : int, optional
        Signal line EMA span (default is 9).

    Returns
    -------
    pd.DataFrame
        DataFrame with MACD, Signal, and Histogram columns added.
    """
    ema_fast = df["Close"].ewm(span=fast, adjust=False).mean()
    ema_slow = df["Close"].ewm(span=slow, adjust=False).mean()

    df["MACD"] = ema_fast - ema_slow
    df["Signal"] = df["MACD"].ewm(span=signal, adjust=False).mean()
    df["Histogram"] = df["MACD"] - df["Signal"]
    return df


In [ ]:
def plot_price_with_emas(df, ticker, save_path=None):
    """
    Plots closing price with EMA20, EMA50, and EMA200.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing price and EMA data.
    ticker : str
        Stock symbol for labeling the chart.
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    """
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.plot(df.index, df["Close"], label="Close", linewidth=1.5)
    ax.plot(df.index, df["EMA20"], label="EMA 20", linewidth=1.0)
    ax.plot(df.index, df["EMA50"], label="EMA 50", linewidth=1.0)
    ax.plot(df.index, df["EMA200"], label="EMA 200", linewidth=1.0)

    ax.set_title(f"{ticker} – Price with EMA20 / EMA50 / EMA200")
    ax.legend()
    ax.grid(True)

    plt.show()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")


In [ ]:
def plot_macd(df, ticker, save_path=None):
    """
    Plots MACD, Signal line, and Histogram.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing MACD data.
    ticker : str
        Stock symbol for labeling the chart.
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    """
    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(df.index, df["MACD"], label="MACD", linewidth=1.5)
    ax.plot(df.index, df["Signal"], label="Signal", linewidth=1.0)
    ax.bar(df.index, df["Histogram"], width=1.0, label="Histogram")

    ax.set_title(f"{ticker} – MACD")
    ax.legend()
    ax.grid(True)

    plt.show()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")


## Example Usage

We now apply the EMA and MACD analysis to the selected stock and visualize the results.


In [ ]:
# Load data
df10 = download_price_data(TICKER, period="3y")

# Calculate indicators
df10 = calculate_ema(df10, spans=(20, 50, 200))
df10 = calculate_macd(df10)

In [ ]:


# Plot and save figures
plot_price_with_emas(
    df10,
    ticker=TICKER,
    save_path=f"figures/{TICKER}_price_with_emas.png"
)

plot_macd(
    df10,
    ticker=TICKER,
    save_path=f"figures/{TICKER}_macd.png"
)
